In [1]:
import pandas as pd
import numpy as np
import os
import shutil
import re
from pathlib import Path

In [2]:
def pivot_table(df, index_col, col, val):
    temp_df = df.copy()
    temp_df["Elapsed Minutes"] = pd.to_timedelta(temp_df['Elapsed Time'])
    temp_df["Elapsed Minutes"] = (temp_df['Elapsed Minutes'].dt.total_seconds() / 60).round(2)
    pivot_table_df = pd.pivot_table(temp_df, index=index_col, columns=col, values=val, aggfunc="mean", fill_value=0).stack().reset_index()
    if isinstance(index_col, list):
        group_cols = index_col + [col]
        new_cols = index_col + [col, "Average Elapsed Minutes"]
    else:
        group_cols = [index_col, col]
        new_cols = [index_col, col, "Average Elapsed Minutes"]
    pivot_table_df.columns = new_cols
    pivot_table_df["Average Elapsed Minutes"] = pivot_table_df["Average Elapsed Minutes"].round(2)
    temp_df["Total Failed Jobs"] = (temp_df["Status Code"] > 1).astype(int)
    temp_df["Total Successful Jobs"] = (temp_df["Status Code"] <= 1).astype(int)
    job_status_summary = temp_df.groupby(index_col + [col])[["Failed Jobs", "Successful Jobs"]].sum().reset_index()
    job_latest_success = (temp_df[temp_df["Status Code"] <= 1].groupby(group_cols)["End Time"].max().reset_index().rename(columns={"End Time": "Latest Successful End Time"}))
    pivot_table_df = pivot_table_df.merge(job_status_summary, on=index_col + [col], how="left")
    pivot_table_df = pivot_table_df.merge(job_latest_success, on=group_cols, how="left")
    return pivot_table_df

In [6]:
policies_list_df = pd.read_csv("data/email_dl.csv")

In [14]:
# source_folders = [Path(r"C:\Users\User\Desktop\Projects\netbackup_weekly_report\data\May_2026")]
source_folders = [Path(r"/Users/willis/Projects/netbackup_weekly_report/data/May_2026")]

all_dfs = []
all_failed_dfs = []

for source_folder in source_folders:
    report_csv_files = source_folder.rglob("daily_backup_report_*")
    failed_report_csv_files = source_folder.rglob("daily_baselined_failed_backup_report_*")
    for report_csv_file in report_csv_files:
        try:
            df = pd.read_csv(report_csv_file)
            df = df.drop(columns = 'Number of Child Jobs')
            if 'Failure Count' in df.columns and 'Backup Selection' in df.columns:
                df = df.drop(columns=['Failure Count', 'Backup Selection'], errors='ignore')
            all_dfs.append(df)
        except Exception as e:
            print(f"Error : {e}")
    for failed_report_csv_file in failed_report_csv_files:
        try:
            df = pd.read_csv(failed_report_csv_file)
            # df = df.drop(columns = 'Number of Child Jobs')
            if 'Backup Selection' in df.columns:
                df = df.drop(columns='Backup Selection', errors='ignore')
            all_failed_dfs.append(df)
        except Exception as e:
            print(f"Error : {e}")

# combine all csv into one dataframe
may_df = pd.concat(all_dfs, ignore_index=True)
may_df["Start Time"] = pd.to_datetime(may_df["Start Time"], errors="coerce")
may_df["End Time"] = pd.to_datetime(may_df["End Time"], errors="coerce")

# combine all csv into one dataframe
may_failed_df = pd.concat(all_failed_dfs, ignore_index=True)
may_failed_df["Start Time"] = pd.to_datetime(may_failed_df["Start Time"], errors="coerce")
may_failed_df["End Time"] = pd.to_datetime(may_failed_df["End Time"], errors="coerce")

In [15]:
may_df

,Job,Type,Schedule,Parent Job,Client,Policy,State,Status Code,Start Time,End Time,Elapsed Time,Backup Selection
0,8800000,BACKUP,Monthly,0,APP-SRV-01,EXCH-DAILY,DONE,0,2026-04-19 20:36:00,2026-04-19 23:32:44,2:56:44,NaN
1,8800001,BACKUP,Monthly,0,EXCH-SRV-01,WEB-DAILY,DONE,13,2026-04-11 03:17:00,2026-04-11 09:42:05,6:25:05,NaN
2,8800002,BACKUP,Daily,0,EXCH-SRV-01,WEB-DAILY,DONE,1,2026-04-27 10:49:00,2026-04-27 11:56:36,1:07:36,NaN
3,8800003,BACKUP,Daily,0,APP-SRV-02,DB-DAILY,DONE,1,2026-04-28 15:24:00,2026-04-28 19:22:14,3:58:14,NaN
4,8800004,BACKUP,Monthly,0,DB-SRV-01,EXCH-DAILY,DONE,0,2026-04-22 07:14:00,2026-04-22 12:49:42,5:35:42,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
416,9100216,BACKUP,Weekly,-,EXCH-SRV-01,EXCH-DAILY,DONE,0,2026-05-09 13:54:21,2026-05-09 15:03:21,1:09:00,C:\
417,9100217,BACKUP,Weekly,-,EXCH-SRV-01,EXCH-DAILY,DONE,10,2026-05-10 11:22:48,2026-05-10 11:55:48,0:33:00,E:\
418,9100218,BACKUP,Daily,-,EXCH-SRV-01,EXCH-DAILY,DONE,10,2026-05-10 19:40:44,2026-05-10 21:21:44,1:41:00,D:\
419,9100219,BACKUP,Weekly,-,EXCH-SRV-01,EXCH-DAILY,DONE,1,2026-05-10 12:16:25,2026-05-10 14:43:25,2:27:00,E:\


In [ ]:
parent_jobs = set(may_failed_df["Job"].astype(str))

may_failed_filtered_df = may_failed_df[
    ~may_failed_df["Parent Job"].astype(str).isin(parent_jobs)
].copy()

may_failed_filtered_df["Start Day"] = may_failed_filtered_df["Start Time"].dt.date
may_failed_filtered_df = may_failed_filtered_df.sort_values(["Client", "Policy", "Start Time"])

may_failed_filtered_df = may_failed_filtered_df.drop_duplicates(subset=["Start Day", "Policy", "Client"], keep="first")

may_failed_filtered_df

In [ ]:
may_failed_filtered_df.loc[may_failed_filtered_df['Client'] == 'MSG-EXDAG-01.dsta.gov.sg']

In [ ]:
may_failed_filtered_df_copy = may_failed_filtered_df.copy()
may_failed_filtered_df_copy = may_failed_filtered_df_copy.sort_values(["Client", "Policy", "Start Time"])

may_failed_filtered_df_copy["New Incident"] = (
    may_failed_filtered_df_copy.groupby(["Client", "Policy"])["Failure Count"]
      .diff()
      .ne(1)
)

may_failed_filtered_df_copy["Incident ID"] = (
    may_failed_filtered_df_copy.groupby(["Client", "Policy"])["New Incident"]
      .cumsum()
)

may_failed_filtered_df_copy

In [ ]:
may_failed_filtered_df_copy.loc[may_failed_filtered_df_copy['Client'] == 'MSG-EXDAG-01.dsta.gov.sg']

In [ ]:
incident_summary = (
    may_failed_filtered_df_copy.groupby(["Client", "Policy", "Incident ID"])
      .agg(
          Start=("Start Time", "min"),
          End=("Start Time", "max"),
          MaxFailureCount=("Failure Count", "max")
      )
      .reset_index()
)

incident_summary

In [ ]:
incident_summary.loc[incident_summary['Client'] == 'MSG-EXDAG-01.dsta.gov.sg']

In [ ]:
incident_summary.to_csv("failed_streak_may_2.csv", index=False)

In [ ]:
policy_df = {}
policy_summary_df = {}

output_path = Path("report_output/")
if not output_path.exists():
    output_path.mkdir()

for pattern in policies_list_df['Policy'].dropna().unique():
    filtered_df_all = may_df[
        may_df['Policy'].astype(str).str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )].copy().sort_values(by=['Client', 'Policy'], ascending=True)
    filtered_df = filtered_df_all.loc[filtered_df_all['Parent Job'] == '-']
    filtered_df = filtered_df.sort_values(by=['Client', 'Policy'], ascending=True)
    filtered_pivot_df = pivot_table(filtered_df.copy(), index_col=["Client", "Policy"], col="State", val="Elapsed Minutes")
    policy_df[pattern] = filtered_df_all
    policy_summary_df[pattern] = filtered_pivot_df

In [ ]:
# for name, df in policy_df.items():
#     df.to_csv(f"{output_path}/{name}_report.csv", index=False)

for name in policy_df.keys():
    destination_file = f"{output_path}/{name}_report.xlsx"
    if os.path.exists(destination_file):
            os.remove(destination_file)
    with pd.ExcelWriter(destination_file, engine="openpyxl") as writer:

        policy_df[name].to_excel(
            writer,
            sheet_name=f"{name}_Report",
            index=False
        )

        policy_summary_df[name].to_excel(
            writer,
            sheet_name=f"{name}_Summary",
            index=False
        )